# ⚛️ QuantumShield-FL: QKD Tutorial

This notebook demonstrates the Quantum Key Distribution (QKD) capabilities of QuantumShield-FL,
including the BB84 and E91 protocols with interactive visualizations.

## Table of Contents
1. [Introduction to QKD](#intro)
2. [BB84 Protocol Simulation](#bb84)
3. [Eavesdropping Detection](#eve)
4. [E91 Protocol & Bell Inequality](#e91)
5. [Key Rate Analysis](#keyrate)
6. [QKD Session for Encrypted Communication](#session)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

from src.quantum.qkd import BB84Protocol, E91Protocol, QKDSession

# Plot style
plt.rcParams['figure.facecolor'] = '#0a0a0a'
plt.rcParams['axes.facecolor'] = '#111111'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.color'] = 'white'
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['grid.color'] = '#222222'

print('✅ QuantumShield-FL loaded successfully!')

## 1. BB84 Protocol Simulation

The BB84 protocol (Bennett & Brassard, 1984) was the first quantum key distribution protocol.
It uses the polarization of photons to transmit key bits, exploiting the no-cloning theorem
to detect eavesdropping.

**Key Steps:**
1. Alice prepares qubits in random bases {+, ×} and random bit values {0, 1}
2. Alice sends qubits to Bob
3. Bob measures in random bases
4. Alice and Bob compare bases over classical channel (sifting)
5. Where bases match → correlated key bits
6. QBER estimation and privacy amplification

In [ ]:
# ─── Run BB84 Without Eavesdropper ───
print('🔐 Running BB84 Protocol (No Eavesdropper)...')
bb84 = BB84Protocol(n_qubits=2048, error_threshold=0.11)
result_no_eve = bb84.run_full_protocol(eavesdrop=False)

print(f"📊 Raw bits:        {result_no_eve['n_qubits']}")
print(f"📊 Sifted key bits: {result_no_eve.get('sifted_key_length', 'N/A')}")
print(f"📊 Final key bits:  {result_no_eve['key_length']}")
print(f"📊 QBER:            {result_no_eve['qber']:.4f} ({result_no_eve['qber']*100:.2f}%)")
print(f"📊 Key rate:        {result_no_eve['key_rate']:.4f} bits/qubit")
print(f"🔒 Secure:          {'✅ YES' if result_no_eve['is_secure'] else '❌ NO'}")

In [ ]:
# ─── Run BB84 With Eavesdropper ───
print('⚠️  Running BB84 Protocol (With Eavesdropper Eve)...')
result_with_eve = bb84.run_full_protocol(eavesdrop=True)

print(f"📊 QBER with Eve:   {result_with_eve['qber']:.4f} ({result_with_eve['qber']*100:.2f}%)")
print(f"🔒 Secure:          {'✅ YES' if result_with_eve['is_secure'] else '❌ NO - Eve detected!'}")

In [ ]:
# ─── QBER vs Key Rate Analysis ───
qber_values = np.linspace(0, 0.25, 100)

def binary_entropy(p):
    if p <= 0 or p >= 1:
        return 0
    return -p * np.log2(p) - (1-p) * np.log2(1-p)

# BB84 secure key rate (Shor-Preskill formula)
secure_rate = np.array([max(0, 1 - 2*binary_entropy(e)) for e in qber_values])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('BB84 Protocol Analysis', color='white', fontsize=14, fontweight='bold')

# Plot 1: Key rate vs QBER
ax = axes[0]
ax.plot(qber_values * 100, secure_rate, color='#00ff88', linewidth=2.5, label='Secure Key Rate')
ax.axvline(x=11, color='#ff4444', linestyle='--', linewidth=1.5, label='Security threshold (11%)')
ax.axvline(x=result_no_eve['qber']*100, color='#44aaff', linestyle='-', linewidth=2, label=f"No Eve: {result_no_eve['qber']*100:.1f}%")
ax.axvline(x=result_with_eve['qber']*100, color='#ff8844', linestyle='-', linewidth=2, label=f"With Eve: {result_with_eve['qber']*100:.1f}%")
ax.fill_between(qber_values*100, secure_rate, where=(qber_values < 0.11), alpha=0.2, color='#00ff88')
ax.set_xlabel('QBER (%)')
ax.set_ylabel('Secure Key Rate (bits/qubit)')
ax.set_title('Key Rate vs QBER', color='white')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Plot 2: Key sizes comparison
ax = axes[1]
categories = ['Raw Qubits', 'Sifted Key', 'Final Key']
no_eve_vals = [
    result_no_eve['n_qubits'],
    result_no_eve.get('sifted_key_length', result_no_eve['key_length'] * 2),
    result_no_eve['key_length']
]
with_eve_vals = [
    result_with_eve['n_qubits'],
    result_with_eve.get('sifted_key_length', result_with_eve['key_length'] * 2),
    result_with_eve['key_length']
]
x = np.arange(len(categories))
width = 0.35
bars1 = ax.bar(x - width/2, no_eve_vals, width, label='No Eve', color='#00ff88', alpha=0.8)
bars2 = ax.bar(x + width/2, with_eve_vals, width, label='With Eve', color='#ff4444', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylabel('Number of Bits')
ax.set_title('Key Generation Statistics', color='white')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../experiments/bb84_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot saved!')

## 2. E91 Protocol — Entanglement-Based QKD

The E91 protocol (Ekert, 1991) uses entangled Bell pairs. Security is guaranteed by
testing the **CHSH inequality** — any eavesdropping reduces the Bell inequality violation.

In [ ]:
print('🔐 Running E91 Protocol...')
e91 = E91Protocol(n_pairs=2000)

result_e91_clean = e91.run_full_protocol(eavesdrop=False)
result_e91_eve = e91.run_full_protocol(eavesdrop=True)

print(f"\n📊 E91 Results (No Eve):")
print(f"   CHSH value: |S| = {abs(result_e91_clean['chsh_value']):.4f} (Tsirelson bound: 2√2 ≈ {2*np.sqrt(2):.4f})")
print(f"   Key length: {result_e91_clean['key_length']} bits")
print(f"   Bell inequality violated: {'✅ YES' if abs(result_e91_clean['chsh_value']) > 2 else '❌ NO'}")

print(f"\n📊 E91 Results (With Eve):")
print(f"   CHSH value: |S| = {abs(result_e91_eve['chsh_value']):.4f}")
print(f"   Eve detected: {'✅ YES' if abs(result_e91_eve['chsh_value']) <= 2 else 'Not detected'}")

## 3. Quantum-Secured Encrypted Communication

In [ ]:
# Full QKD session with AES-256 encryption
print('🔐 Establishing Quantum-Secured Communication Session...')
session = QKDSession(protocol='bb84', n_qubits=2048)
session.establish_key()

# Encrypt model gradient
import json
gradient = {'layer_0': [0.123, -0.456, 0.789], 'layer_1': [-0.321, 0.654]}
plaintext = json.dumps(gradient).encode()

ciphertext = session.encrypt(plaintext)
decrypted = session.decrypt(ciphertext)

print(f"\n📦 Original gradient: {gradient}")
print(f"🔒 Encrypted (hex):   {ciphertext[:32].hex()}...")
print(f"🔓 Decrypted:         {json.loads(decrypted)}")
print(f"✅ Round-trip verified: {plaintext == decrypted}")